# nano Beatrix — the far read: sixteen bound pairs past a million keys, lane 1 of 3

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_capacity_far_lane1.ipynb)

**The question**: Six sixteen-pair models with the records' keys held at one length make no wrong look at a million competing keys; their six twins without the hold make up to nine in 4,608. Where do sixteen held pairs start to fail, and where do half the questions go wrong?

**This notebook** trains nothing. It reads each model's capacity the way the million-key read did (the same 192 rows a model, half with key-hard keys; the same kind of bank, each competitor a stored end-marker's content turned by a fresh key's phases), now to 10^8 competitors on a grid (10^3, 4,096, 10^4, 10^5, 10^6, 2x10^6, 5x10^6, 10^7, 2x10^7, 5x10^7, 10^8): the leak and the look error three ways on the same competitors (as read; with every competitor's pair lengths set to its entry's true record's; with every true score set to the model's median), beside the random-phase moment N I0(s)^J / (exp(J s) + N I0(s)^J), the first N with a wrong look and the N where half the looks go wrong. The bank is streamed in blocks (the same numbers as the million-key read's, without holding 10^8 vectors); the twelve models share one bank of keys, drawn once a lane.

**The runtime**: a TPU runtime (Runtime > Change runtime type > v6e-1 TPU) for its host machine's many cores and large memory; the read runs on the CPU with every core. A CPU runtime with High-RAM reads the same, slower.

**The lanes**: three lanes, each on its own runtime. Every lane first reproduces the stored million-key read of the first model from the same bank (the replication gate: its numbers must equal the stored record's) and prices a model at 10^8 from the gate's walls; then it walks the twelve models (the held six first, then the twins), claiming each before reading it by a small file under `reports/capacity_far/claims/` of the rotary repo, written only if the repo has not moved since the lane's read of it, so no two lanes read one model. Each record ships to `reports/capacity_far/` as it lands.

**The cells**: C1 pins the trainer and holds the decision block and the lane's name; C2 writes the held-out documents (the rows' filler), lists the models and the records and probes the push path; C3 is the replication gate and the price line; C4 is the lane; C5 renders the tables of the records this lane read.

**Before C1**: Runtime > Change runtime type > v6e-1 TPU. The notebook's `HF_TOKEN` secret (write access to [AbstractPhil/geolip-rotary-experiments](https://huggingface.co/AbstractPhil/geolip-rotary-experiments)) with notebook access on: a lane claims before it reads, so a tokenless lane reads nothing. Run C1, then Runtime > Restart session and run all. A file named `STOP` in `/content/out` starts no new model; a restarted session takes back its own unfinished claim. The code and its tests: [AbstractEyes/geolip-nano-beatrix-trainer](https://github.com/AbstractEyes/geolip-nano-beatrix-trainer) (`geolip/nano_beatrix/capacity_far.py`, `tests/test_capacity_far.py`).

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK (a TPU runtime's host; the read runs on the CPU). RESTART the session after the first run of this cell (Runtime > Restart session and run all).
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.89" "amoe-lora @ git+https://github.com/AbstractEyes/amoe-lora@v0.2.12"
import torch, amoe
import geolip.nano_beatrix as N
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 89), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart session, then rerun from C1')
assert amoe.__version__ == '0.2.12', 'the arm module of record is amoe-lora 0.2.12'
torch.set_num_threads(os.cpu_count() or 2)
_mem = [int(l.split()[1]) for l in open('/proc/meminfo') if l.startswith('MemTotal')]
print('geolip.nano_beatrix', N.__version__, '| amoe', amoe.__version__, '| torch', torch.__version__, '| host CPUs', os.cpu_count(),
      '| cpu threads', torch.get_num_threads(), '| memory', f"{_mem[0] / 2 ** 20:.0f} GB" if _mem else '?')

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN            # the shipper reads it from the environment; the value is never printed

# ============ THE DECISION BLOCK — every setting from the plan, with its source beside it ============
OUT = '/content/out'                                   # the session folder: the held-out documents, the records, a model's weights while it is read
REPO_W = 'AbstractPhil/alephllm-nano-beatrix-training' # the models' weights and their stored million-key records are read from here
REPO = 'AbstractPhil/geolip-rotary-experiments'       # the rotary experiments' repo: every far record and claim ships here
DEST = 'reports/capacity_far'                          # the records' folder on the rotary repo
CLAIMS = 'reports/capacity_far/claims'                 # the lanes' claims
R = 192                                                # rows a model, half key-hard: the million-key read's
SEED = 1                                               # the rows' generator and the bank's: the million-key read's
from geolip.nano_beatrix import capacity as _cp, capacity_far as _cf
NS = _cf.FAR_NS                                        # 10^3, 4,096, 10^4, 10^5, 10^6, 2x10^6, 5x10^6, 10^7, 2x10^7, 5x10^7, 10^8
BLOCK = _cf.BLOCK                                      # the bank streamed 65,536 competitors at a time
TAIL = '_L24000_ctx4096_frdyn_cap_mode_sz32-128-256-512_r4_n30_o0_askany_gag50'
FAM = {'L16R': '_cp4_gc48_kh50_va1_pt10b4h0_krt16r218_ahdigit1_rec16',   # 48 records, the key-hard rows, sixteen bound pairs, the records held
       'L16': '_cp4_gc48_kh50_va1_pt10b4h0_krt16_ahdigit1_norm16'}      # the same without the hold (the twins)
CORES = [(f'{f} s{s}', f'alhena-both_s{s}{TAIL}{FAM[f]}') for f in ('L16R', 'L16') for s in range(1, 7)]   # the held six first
GATE = CORES[0]                                        # the replication gate: L16R s1's stored million-key read, reproduced from the same bank
STOP_MIN = 60                                          # the price line's stop: a model projected over 60 min at 10^8
LANE = 'lane1'                                      # this lane's name: its claims carry it
print('decision block:', LANE, '|', len(CORES), 'models |', R, 'rows a model, seed', SEED, '| N to', f"{NS[-1]:,}", '| ships to',
      REPO + '/' + DEST if HF_TOKEN else 'nothing (tokenless)')

In [ ]:
# C2 (P) — PREFLIGHT: the held-out documents (the rows' filler), the models and the records on the Hub, the push probe. Nothing is read yet.
import json, os, time
from huggingface_hub import HfApi
from geolip.nano_beatrix import bed_corpus
HELD = bed_corpus.write_heldout(f'{OUT}/bed')         # the first 2,048 documents of fineweb-edu sample-10BT (the beds' held-out head)
print('held-out documents', HELD, os.path.getsize(HELD) / 1e6, 'MB')
api = HfApi(token=HF_TOKEN or False)
FILES_W = set(api.list_repo_files(REPO_W))
missing = [lab for lab, tag in CORES if not all(f'bed/weights/{tag}/{f}' in FILES_W for f in ('model.safetensors', 'key_rot.safetensors', 'scale.json'))]
assert not missing, ('models not on the Hub', missing)
GATE_TAIL = f'bed/reports/capacity_tail/{_cp.tail_name(GATE[1], R, SEED)}'
assert GATE_TAIL in FILES_W, ('the gate needs the stored million-key record', GATE_TAIL)
FILES = set(api.list_repo_files(REPO))
done_hub = [lab for lab, tag in CORES if f'{DEST}/{_cf.far_name(tag, R, SEED)}' in FILES]
print('the twelve models on the Hub | records already shipped (done before):', done_hub or 'none')
PUSH_OK = False
if HF_TOKEN:
    path = f'{DEST}/preflight/push_probe_{LANE}.json'
    body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'host_cpus': os.cpu_count(), 'cpu_threads': torch.get_num_threads(),
                       'models': [lab for lab, _ in CORES], 'rows': R, 'grid': list(NS)}).encode()
    api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message=f'nano capacity far: push probe ({LANE})')
    PUSH_OK = path in set(api.list_repo_files(REPO))
    print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
else:
    print('TOKENLESS: no probe written; the lane reads and ships nothing')

In [ ]:
# C3 (G) — THE REPLICATION GATE AND THE PRICE LINE: the stored million-key read of the first model reproduced from the same bank (every
# statistic within 1e-5 relative, the median-score condition within 1e-4, the look and argmax errors, the band counts and the strongest
# competitors exact), shipped to reports/capacity_far/gate/; then a model at 10^8 priced from the gate's walls (the scores scale with N, the
# bank is drawn once a lane). A gate that fails or a price over STOP_MIN starts no lane.
import json, os, time
from huggingface_hub import hf_hub_download, snapshot_download
from geolip.nano_beatrix import capacity as cp, capacity_far as cf, rotary_scale as rs
held = rs.load_heldout(HELD)
label, tag = GATE
snapshot_download(REPO_W, allow_patterns=[f'bed/weights/{tag}/*'], local_dir=f'{OUT}/w', token=HF_TOKEN or None)
stored = json.load(open(hf_hub_download(REPO_W, GATE_TAIL, token=HF_TOKEN or None), encoding='utf-8'))
t0 = time.time()
gate = cf.far_core(f'{OUT}/w/bed/weights/{tag}', held, R=R, seed=SEED, ns=cp.TAIL_NS, block=BLOCK, log=print)
bad = cf.compare_tail(gate, stored)
GATE_OK = not bad
os.makedirs(f'{OUT}/gate', exist_ok=True)
gpath = f"{OUT}/gate/{cf.far_name(tag, R, SEED).replace('_capfar_', '_capfar_gate_')}"
json.dump(dict(gate, gate=dict(ok=GATE_OK, mismatches=[str(b) for b in bad[:50]], stored=GATE_TAIL)), open(gpath, 'w', encoding='utf-8'), indent=1)
if HF_TOKEN and PUSH_OK:
    api.upload_file(path_or_fileobj=gpath, path_in_repo=f'{DEST}/gate/{LANE}_{os.path.basename(gpath)}', repo_id=REPO,
                    commit_message=f'nano capacity far: the gate ({LANE}) {"MATCH" if GATE_OK else "MISMATCH"}')
w = gate['wall']
per = NS[-1] / max(cp.TAIL_NS)                         # the scores grow with N (the bank's blocks); the capture does not
proj = w['capture'] + w['bank'] * per + w['scores'] * per
proj_next = w['capture'] + w['scores'] * per            # a model after the first in a lane: the bank's phases cached
print(f"THE GATE: {'MATCH' if GATE_OK else 'MISMATCH'} ({len(bad)} differences) | walls: capture {w['capture']:.0f} s, bank {w['bank']:.0f} s, "
      f"scores {w['scores']:.0f} s at 10^6 | a model at 10^8: ~{proj / 60:.0f} min the first, ~{proj_next / 60:.0f} min the next "
      f"(the stop at {STOP_MIN} min)")
if bad:
    print('the first differences:', bad[:10])
assert GATE_OK, 'the replication gate failed: no lane starts'
assert proj_next / 60 <= STOP_MIN, 'the price line is over the stop: no lane starts'

In [ ]:
# C4 (R) — THE LANE: the twelve models in order (the held six, then the twins), one at a time. Before each model the rotary repo is read at
# one commit: a model with its record there (or in OUT/capacity_far) is done; a model with another lane's claim younger than FRESH_S is
# taken; the first model with neither is claimed (a small file under CLAIMS, written only if the repo has not moved since that read - else
# the read repeats), read to 10^8, and its record shipped as it lands (a background thread with retries; the local record is the done mark).
# The bank's phases are drawn once and kept (CACHE). A STOP file in OUT starts no new model. The lane ends when no model is free. The job's
# size, the time spent and the time left print after every model.
import json, os, random, shutil, threading, time
from huggingface_hub import snapshot_download
from geolip.nano_beatrix import capacity_far as cf
assert HF_TOKEN and PUSH_OK and GATE_OK, 'a lane claims before it reads: the token, the push probe (C2) and the gate (C3)'
REC = f'{OUT}/capacity_far'
FRESH_S = 3 * 3600                                     # a claim this old with no record is a stopped lane's: the model is free again
CACHE = {}
os.makedirs(REC, exist_ok=True)
threads, failed = [], []

def _ship(path):
    for i in range(6):
        try:
            api.upload_file(path_or_fileobj=path, path_in_repo=f'{DEST}/{os.path.basename(path)}', repo_id=REPO,
                            commit_message=f'nano capacity far: {os.path.basename(path)[:90]}')
            return
        except Exception as e:
            print('ship retry', i + 1, os.path.basename(path)[:60], type(e).__name__)
            time.sleep(min(300, 10 * 2 ** i))
    failed.append(path)

def ship(path):
    t = threading.Thread(target=_ship, args=(path,))
    t.start()
    threads.append(t)

def snap():
    # the repo's head commit and its files, read at that one commit
    sha = api.repo_info(REPO).sha
    return sha, set(api.list_repo_files(REPO, revision=sha))

def done(tag, files):
    name = cf.far_name(tag, R, SEED)
    return f'{DEST}/{name}' in files or os.path.exists(f'{REC}/{name}')

def claims(label, files):
    # the fresh claims on a model, [(lane, utc seconds)], from the claim files' names: <label>__<lane>__<utc seconds>.json
    pre = f"{CLAIMS}/{label.replace(' ', '_')}__"
    out = []
    for f in files:
        if f.startswith(pre) and f.endswith('.json'):
            lane, utc = f[len(pre):-len('.json')].split('__')
            if time.time() - int(utc) < FRESH_S:
                out.append((lane, int(utc)))
    return out

def pick():
    # -> (label, tag) claimed by this lane, or None when no model is left
    for attempt in range(40):
        sha, files = snap()
        for label, tag in CORES:                       # this lane's own claim on an unfinished model (a restarted session) is still its
            if not done(tag, files) and any(lane == LANE for lane, _ in claims(label, files)):
                return label, tag
        left = [(label, tag) for label, tag in CORES if not done(tag, files) and not claims(label, files)]
        if not left:
            return None
        label, tag = left[0]
        path = f"{CLAIMS}/{label.replace(' ', '_')}__{LANE}__{int(time.time())}.json"
        try:
            api.upload_file(path_or_fileobj=json.dumps({'lane': LANE, 'model': tag}).encode(), path_in_repo=path, repo_id=REPO,
                            commit_message=f'nano capacity far: {LANE} claims {label}', parent_commit=sha)
            return label, tag
        except Exception as e:                          # the repo moved since the read (another lane's claim or record) or the call failed:
            print('claim retry', attempt + 1, label, type(e).__name__)   # read again (a claim that landed anyway is found as this lane's)
            time.sleep(random.uniform(3, 12))
    raise RuntimeError('no claim after 40 tries')

def fmt(n):
    return 'none on the grid' if n is None else f'{n:,.0f}'

_, FILES = snap()
for f in sorted(os.listdir(REC)):                     # local records not on the Hub: shipped first
    if f.endswith('.json') and f'{DEST}/{f}' not in FILES:
        ship(f'{REC}/{f}')
t0, read_n, recs = time.time(), 0, {}
while True:
    if os.path.exists(f'{OUT}/STOP'):
        print('STOP present: no new model starts')
        break
    got = pick()
    if got is None:
        print('no model is left: the lane ends')
        break
    label, tag = got
    local = f'{REC}/{cf.far_name(tag, R, SEED)}'
    t1 = time.time()
    snapshot_download(REPO_W, allow_patterns=[f'bed/weights/{tag}/*'], local_dir=f'{OUT}/w', token=HF_TOKEN)
    cf.run_far(f'{OUT}/w/bed/weights/{tag}', held, REC, R=R, seed=SEED, ns=NS, block=BLOCK, cache=CACHE, log=print)
    ship(local)
    if tag != GATE[1]:
        shutil.rmtree(f'{OUT}/w/bed/weights/{tag}', ignore_errors=True)
    read_n += 1
    rec = recs[label] = json.load(open(local, encoding='utf-8'))
    b, fb = rec['base'], rec['far']['base']
    _, files = snap()
    open_n = sum(not done(tag_, files) for _, tag_ in CORES)
    spent = time.time() - t0
    print(f"{label} [{LANE}]: s {rec['s_core']:.2f} | look error base at 10^6 {b['1000000']['look_error']:.4f}, 10^7 "
          f"{b['10000000']['look_error']:.4f}, 10^8 {b['100000000']['look_error']:.4f} | the first wrong look at {fmt(fb['onset'])}, half "
          f"wrong at {fmt(fb['half'])} | {time.time() - t1:.0f} s | this lane: {read_n} read in {spent / 60:.1f} min | {open_n} of "
          f"{len(CORES)} models without a record, ~{open_n * spent / read_n / 3 / 60:.0f} min left at 3 lanes")
for t in threads:
    t.join()
_, FILES = snap()
print('this lane read', read_n, '| UNSHIPPED:', [f for f in sorted(os.listdir(REC)) if f.endswith('.json') and f'{DEST}/{f}' not in FILES])

In [ ]:
# C5 — THE TABLES (the records this lane read): per model the base look error along the grid, the first wrong look and the half point
# (three ways), the mean leak over the moment at 10^6 / 10^7 / 10^8, and where the base look errors' strongest competitors come from at 10^8.
from IPython.display import Markdown, display
grid = ('1000000', '10000000', '20000000', '50000000', '100000000')
rows = ['| model | s | base look error at 10^6 / 10^7 / 2x10^7 / 5x10^7 / 10^8 | first wrong look: base / M / E | half wrong: base / M / E | '
        'mean / moment at 10^6 / 10^7 / 10^8 | strongest competitor at 10^8: own record / its row / another row / an index entry |',
        '|---|---|---|---|---|---|---|']
fm = lambda n: '-' if n is None else f'{n:.2e}'   # noqa: E731
for label, tag in CORES:
    if label not in recs:
        continue
    r = recs[label]
    le = ' / '.join(f"{r['base'][n]['look_error']:.4f}" for n in grid)
    on = ' / '.join(fm(r['far'][c]['onset']) for c in ('base', 'M', 'E'))
    hf = ' / '.join(fm(r['far'][c]['half']) for c in ('base', 'M', 'E'))
    mm = ' / '.join(f"{r['base'][n]['mean'] / r['base'][n]['moment']:.2f}" for n in ('1000000', '10000000', '100000000'))
    src = ' / '.join(str(v) for v in r['top_competitor']['source'].values())
    rows.append(f"| {label} | {r['s_core']:.2f} | {le} | {on} | {hf} | {mm} | {src} |")
display(Markdown('\n'.join(rows)))